# Linked Lists, Stacks & Queues — Senior Interview Patterns

Adobe Interview Prep | 14 YOE

Covers:

  1. Singly Linked List — full implementation

  2. Reverse a linked list (iterative & recursive)

  3. Detect & find cycle (Floyd's tortoise-hare)

  4. Merge two sorted lists

  5. Merge K sorted lists (heap)

  6. Find middle node (fast/slow pointer)

  7. LRU Cache (OrderedDict + doubly-linked list)

  8. Min Stack (O(1) getMin)

  9. Monotonic Stack — next greater element, largest rectangle in histogram

  10. Deque-based Sliding Window Maximum

In [ ]:
from __future__ import annotations
from collections import OrderedDict, deque
from heapq import heappush, heappop
from typing import Optional

## 1. Linked List Node & Helpers

**Concept:** Core singly-linked list primitives. `to_list` and `build_list` helpers make testing easy without manually chaining nodes.  

**Use when:** Any linked-list problem — these helpers let you construct and inspect lists quickly.  

**Time:** O(n) for build/to_list &nbsp;|&nbsp; **Space:** O(n)

In [ ]:
class ListNode:
    __slots__ = ("val", "next")

    def __init__(self, val: int = 0, nxt: Optional["ListNode"] = None):
        self.val = val
        self.next = nxt

    def __repr__(self):
        vals, cur = [], self
        while cur:
            vals.append(str(cur.val))
            cur = cur.next
        return " -> ".join(vals)

In [ ]:
def build_list(vals: list[int]) -> Optional[ListNode]:
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head: Optional[ListNode]) -> list[int]:
    out, cur = [], head
    while cur:
        out.append(cur.val)
        cur = cur.next
    return out

# ── demo ──
node = build_list([1, 2, 3, 4, 5])
print(to_list(node))

## 2. Reverse Linked List

**Concept:** Iterative: track `prev/cur/next` pointers, rewire one node at a time. Recursive: reverse the rest, then point the next node back at head.  

**Use when:** In-place reversal, palindrome check, k-group reversal.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1) iterative · O(n) recursive stack

In [ ]:
def reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    Iterative reversal.
    Time O(n)  Space O(1)

    >>> to_list(reverse_list(build_list([1,2,3,4,5])))
    [5, 4, 3, 2, 1]
    """
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

# ── demo ──
print(to_list(reverse_list(build_list([1,2,3,4,5]))))

In [ ]:
def reverse_list_recursive(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    Recursive reversal.
    Time O(n)  Space O(n) stack

    >>> to_list(reverse_list_recursive(build_list([1,2,3])))
    [3, 2, 1]
    """
    if not head or not head.next:
        return head
    new_head = reverse_list_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

# ── demo ──
print(to_list(reverse_list_recursive(build_list([1,2,3]))))

In [ ]:
def reverse_between(
    head: Optional[ListNode], left: int, right: int
) -> Optional[ListNode]:
    """
    Reverse sublist from position left to right (1-indexed).
    Time O(n)  Space O(1)

    >>> to_list(reverse_between(build_list([1,2,3,4,5]), 2, 4))
    [1, 4, 3, 2, 5]
    """
    dummy = ListNode(0, head)
    pre = dummy
    for _ in range(left - 1):
        pre = pre.next
    cur = pre.next
    for _ in range(right - left):
        nxt = cur.next
        cur.next = nxt.next
        nxt.next = pre.next
        pre.next = nxt
    return dummy.next

# ── demo ──
print(to_list(reverse_between(build_list([1,2,3,4,5]), 2, 4)))

## 3. Floyd's Cycle Detection

**Concept:** Slow pointer moves 1 step, fast pointer moves 2. If they meet, a cycle exists. Phase 2: reset one pointer to head; advance both 1 step until they meet — that's the cycle entry.  

**Use when:** Detect cycle in linked list, find cycle start, find duplicate in array (pigeonhole).  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def has_cycle(head: Optional[ListNode]) -> bool:
    """
    Time O(n)  Space O(1)

    >>> n = build_list([3,2,0,-4]); n.next.next.next.next = n.next; has_cycle(n)
    True
    """
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

# ── demo ──
no_cycle = build_list([1,2,3])
print("no cycle:", has_cycle(no_cycle))
cycle_node = build_list([3,2,0,-4])
cycle_node.next.next.next.next = cycle_node.next
print("has cycle:", has_cycle(cycle_node))

In [ ]:
def detect_cycle_start(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    Return the node where the cycle begins, or None.
    Phase 1: detect meeting point.
    Phase 2: move one pointer to head; advance both at speed 1 -> meet at cycle start.
    Time O(n)  Space O(1)
    """
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            slow = head
            while slow is not fast:
                slow = slow.next
                fast = fast.next
            return slow
    return None

# ── demo ──
n = build_list([3,2,0,-4])
n.next.next.next.next = n.next   # cycle at index 1
start = detect_cycle_start(n)
print("cycle start val:", start.val if start else None)

## 4. Find Middle Node

**Concept:** Fast/slow pointers — when fast reaches the end, slow is at the middle. For even-length lists, slow lands on the second middle node.  

**Use when:** Split list in half, palindrome check, merge sort on linked list.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def find_middle(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    Fast/slow pointer — returns second middle for even-length lists.
    Time O(n)  Space O(1)

    >>> find_middle(build_list([1,2,3,4,5])).val
    3
    >>> find_middle(build_list([1,2,3,4])).val
    3
    """
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

# ── demo ──
print(find_middle(build_list([1,2,3,4,5])).val)
print(find_middle(build_list([1,2,3,4])).val)

## 5. Merge Two Sorted Lists

**Concept:** Greedy merge with a dummy head node to avoid edge-case logic. Compare heads, attach the smaller, advance that pointer.  

**Use when:** Merging sorted sequences, merge step in merge sort.  

**Time:** O(m+n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def merge_two_sorted(
    l1: Optional[ListNode], l2: Optional[ListNode]
) -> Optional[ListNode]:
    """
    Time O(n+m)  Space O(1)

    >>> to_list(merge_two_sorted(build_list([1,2,4]), build_list([1,3,4])))
    [1, 1, 2, 3, 4, 4]
    """
    dummy = cur = ListNode()
    while l1 and l2:
        if l1.val <= l2.val:
            cur.next = l1; l1 = l1.next
        else:
            cur.next = l2; l2 = l2.next
        cur = cur.next
    cur.next = l1 or l2
    return dummy.next

# ── demo ──
print(to_list(merge_two_sorted(build_list([1,2,4]), build_list([1,3,4]))))

## 6. Merge K Sorted Lists

**Concept:** Use a min-heap of `(val, list_id, node)` tuples — always extract the smallest current node, push its successor. Avoids O(Nk) naive comparison.  

**Use when:** K-way merge, external sort, stream merging.  

**Time:** O(N log k) where N = total nodes, k = number of lists &nbsp;|&nbsp; **Space:** O(k)

In [ ]:
def merge_k_sorted(lists: list[Optional[ListNode]]) -> Optional[ListNode]:
    """
    Time O(N log k)  Space O(k)  where N = total nodes, k = number of lists.

    >>> lists = [build_list([1,4,5]), build_list([1,3,4]), build_list([2,6])]
    >>> to_list(merge_k_sorted(lists))
    [1, 1, 2, 3, 4, 4, 5, 6]
    """
    heap: list[tuple[int, int, ListNode]] = []
    for i, node in enumerate(lists):
        if node:
            heappush(heap, (node.val, i, node))

    dummy = cur = ListNode()
    while heap:
        val, i, node = heappop(heap)
        cur.next = node
        cur = cur.next
        if node.next:
            heappush(heap, (node.next.val, i, node.next))
    return dummy.next

# ── demo ──
lists = [build_list([1,4,5]), build_list([1,3,4]), build_list([2,6])]
print(to_list(merge_k_sorted(lists)))

## 7. LRU Cache

**Concept:** `OrderedDict` provides O(1) `move_to_end` and `popitem(last=False)`. Manual doubly-linked list + hashmap gives explicit control over pointers for interviewers.  

**Use when:** Cache eviction, any "least recently used" system design.  

**Time:** O(1) get and put &nbsp;|&nbsp; **Space:** O(capacity)

In [ ]:
class LRUCache:
    """
    O(1) get and put using OrderedDict (doubly-linked list + hash map).
    """

    def __init__(self, capacity: int):
        self.cap = capacity
        self._cache: OrderedDict[int, int] = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self._cache:
            return -1
        self._cache.move_to_end(key)
        return self._cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self._cache:
            self._cache.move_to_end(key)
        self._cache[key] = value
        if len(self._cache) > self.cap:
            self._cache.popitem(last=False)

# ── demo ──
lru = LRUCache(2)
lru.put(1, 1); lru.put(2, 2)
print(lru.get(1))       # 1
lru.put(3, 3)           # evicts key 2
print(lru.get(2))       # -1 (evicted)
lru.put(4, 4)           # evicts key 1
print(lru.get(1))       # -1
print(lru.get(3))       # 3
print(lru.get(4))       # 4

## 8. Min Stack

**Concept:** Pair each pushed value with the running minimum at that point. Auxiliary min-stack mirrors the main stack — no extra traversal on `getMin`.  

**Use when:** Stack that supports O(1) minimum query.  

**Time:** O(1) push/pop/top/getMin &nbsp;|&nbsp; **Space:** O(n)

In [ ]:
class MinStack:
    """
    Stack supporting push / pop / top / getMin in O(1).

    >>> ms = MinStack()
    >>> ms.push(-2); ms.push(0); ms.push(-3)
    >>> ms.getMin()
    -3
    >>> ms.pop(); ms.top()
    0
    >>> ms.getMin()
    -2
    """

    def __init__(self):
        self._stack: list[int] = []
        self._min_stack: list[int] = []

    def push(self, val: int) -> None:
        self._stack.append(val)
        m = val if not self._min_stack else min(val, self._min_stack[-1])
        self._min_stack.append(m)

    def pop(self) -> None:
        self._stack.pop()
        self._min_stack.pop()

    def top(self) -> int:
        return self._stack[-1]

    def getMin(self) -> int:
        return self._min_stack[-1]

# ── demo ──
ms = MinStack()
ms.push(-2); ms.push(0); ms.push(-3)
print("getMin:", ms.getMin())   # -3
ms.pop()
print("top:", ms.top())         # 0
print("getMin:", ms.getMin())   # -2

## 9. Monotonic Stack — Next Greater Element

**Concept:** Maintain a decreasing stack of indices. When the current element is greater than `nums[stack[-1]]`, pop and record the answer for that index. Front of stack is always the "waiting" smaller element.  

**Use when:** Next greater/smaller element, daily temperatures, largest rectangle in histogram.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(n)

In [ ]:
def next_greater_element(nums: list[int]) -> list[int]:
    """
    For each element, find the next greater element to the right.
    -1 if none exists.  Uses a decreasing monotonic stack.
    Time O(n)  Space O(n)

    >>> next_greater_element([2, 1, 2, 4, 3])
    [4, 2, 4, -1, -1]
    """
    n = len(nums)
    result = [-1] * n
    stack: list[int] = []
    for i in range(n):
        while stack and nums[i] > nums[stack[-1]]:
            result[stack.pop()] = nums[i]
        stack.append(i)
    return result

# ── demo ──
print(next_greater_element([2, 1, 2, 4, 3]))

## 10. Sliding Window Maximum

**Concept:** Monotonic deque (decreasing) stores indices. Front is always the current window maximum. Pop front when it falls out of window; pop back when new element is larger.  

**Use when:** Maximum (or minimum) in every sliding window of size k.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(k)

In [ ]:
def sliding_window_maximum(nums: list[int], k: int) -> list[int]:
    """
    Max value in each sliding window of size k.
    Monotonic deque (decreasing) — O(n) time, O(k) space.

    >>> sliding_window_maximum([1,3,-1,-3,5,3,6,7], 3)
    [3, 3, 5, 5, 6, 7]
    """
    dq: deque[int] = deque()
    result: list[int] = []
    for i, n in enumerate(nums):
        while dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] < n:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

# ── demo ──
print(sliding_window_maximum([1,3,-1,-3,5,3,6,7], 3))

In [ ]:
if __name__ == "__main__":
    import doctest
    results = doctest.testmod(verbose=False)
    print(f"Linked Lists / Stacks / Queues: {results.attempted} tests, {results.failed} failed")

    lru = LRUCache(2)
    lru.put(1, 1); lru.put(2, 2)
    assert lru.get(1) == 1
    lru.put(3, 3)
    assert lru.get(2) == -1
    print("LRUCache: OK")

    print("All assertions passed.")